# Lab 02 | Mutation detective
**Molecular Cell Biology · 50-minute session · Junior/senior biology majors**

## Goal
Use supplied code to ask: **Does every one-base DNA change alter a predicted protein sequence?**
Build on Lab 01's DNA analysis, now using a small FASTA sequence file.

**One workspace:** stay in Colab. No terminal, GitHub setup, paid plan, AI-generated code, packages, or Drive mounting today.
**One written record:** answer in the Word handout's response sheets (pages 5-6), on paper, or in a plain document using the same headings. Do not type answers into code cells.

**Do not choose Run all before making your predictions.** Work from top to bottom using the numbered cells below.

## Setup
Open your own editable copy. If this is an instructor-shared notebook, use **File → Save a copy in Drive**. If access is blocked, try the downloaded `.ipynb` through **File → Upload notebook**. Use a normal Python 3 CPU runtime. If either route remains blocked after two minutes, tell the instructor and use an approved partner, demonstration, or the paper backup. Never share account credentials.

A **code cell** has a triangular Run button on its left; results appear underneath. Click inside a code cell and press **Shift+Enter** if the button is hard to find. Ignore the AI/chat panel for this exercise.


### Cell 1 — Check the connection
Run without editing. **Expected:** `LAB 02 READY`. An execution count by itself is not the result; read the text underneath the cell.


In [1]:
print("LAB 02 READY")

LAB 02 READY


### Cell 2 — Prepare the file and tools
Run the next cell **without editing**. It contains the supplied analysis tool, not code you need to type or understand line by line today. The notebook creates and reads its own file: you do **not** upload a second data file.

**Expected:** `SETUP COMPLETE: 4 sequences loaded from lab02_sequences.fasta`, followed by four labeled DNA records. On your response sheet, explain what `>A` identifies.

#### Model assumptions
These are **invented, 30-nucleotide coding-strand sequences written 5′ to 3′**, not template strands, whole genes, or patient data. Use reading frame +1 (the first base), the standard genetic code, and the first in-frame stop. For these simplified examples, the RNA text matches the coding DNA except U replaces T. The model does not simulate splicing, RNA degradation, protein folding, or protein function. A stop is a signal, not an amino acid.

The setup cell clears the in-memory run record. After rerunning it, rerun Cells 3-6 with the correct labels before using Cell 7.


In [3]:
#@title 2. Prepare the sequence file and tools (run; do not edit)
# All sequences are invented teaching examples, not patient or organism data.
# Uses only the Python standard library. No installation or Drive access.
from pathlib import Path

FASTA_TEXT = '>control\nATGGCTTTTCAAGAATGGCCTAAAGGCTAA\n>A\nATGGCCTTTCAAGAATGGCCTAAAGGCTAA\n>B\nATGGCTTTTCAAGTATGGCCTAAAGGCTAA\n>C\nATGGCTTTTCAAGAATAGCCTAAAGGCTAA\n'
SEQUENCE_FILE = Path("lab02_sequences.fasta")
SEQUENCE_FILE.write_text(FASTA_TEXT, encoding="utf-8")

# NCBI standard genetic code (table 1). DNA T corresponds to RNA U.
BASES = "TCAG"
AMINO_ACIDS = "FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODONS = [a + b + c for a in BASES for b in BASES for c in BASES]
GENETIC_CODE = dict(zip(CODONS, AMINO_ACIDS))


def read_fasta(path):
    """Read this lab's simple FASTA file; reject invalid or duplicate records."""
    records = {}
    current = None
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith(">"):
            current = line[1:].split()[0] if line[1:].strip() else ""
            if not current or current in records:
                raise ValueError("Missing or duplicate sample label. Rerun Cell 2.")
            records[current] = ""
        else:
            if current is None:
                raise ValueError("A FASTA label must come first. Rerun Cell 2.")
            sequence = "".join(line.split()).upper()
            if set(sequence) - set("ACGT"):
                raise ValueError("This lab accepts only A, C, G, T. Rerun Cell 2.")
            records[current] += sequence
    if not records or any(not sequence for sequence in records.values()):
        raise ValueError("Empty FASTA data. Rerun Cell 2.")
    return records


def translate_coding_dna(sequence):
    """Translate from base 1 in frame +1; stop before the first stop codon."""
    if not sequence or set(sequence) - set("ACGT"):
        raise ValueError("Use nonempty uppercase DNA containing only A, C, G, T.")
    if len(sequence) % 3 != 0:
        raise ValueError("This exercise requires complete triplets; no indels.")
    peptide = []
    for index in range(0, len(sequence), 3):
        codon = sequence[index:index + 3]
        amino_acid = GENETIC_CODE[codon]
        if amino_acid == "*":
            return "".join(peptide), index // 3 + 1, codon.replace("T", "U")
        peptide.append(amino_acid)
    return "".join(peptide), None, None


def triplets(sequence):
    return " ".join(sequence[i:i + 3] for i in range(0, len(sequence), 3))


SEQUENCES = read_fasta(SEQUENCE_FILE)
RUNS = {}  # Recreated by setup; rerun Cells 3-6 after rerunning Cell 2.
assert set(SEQUENCES) == {"control", "A", "B", "C"}
assert len(GENETIC_CODE) == 64
assert translate_coding_dna(SEQUENCES["control"])[0] == "MAFQEWPKG"
for label in ("A", "B", "C"):
    assert len(SEQUENCES[label]) == len(SEQUENCES["control"]) == 30
    assert sum(a != b for a, b in zip(SEQUENCES[label], SEQUENCES["control"])) == 1


def analyze(sample_id):
    """Display one sample, compare it with control, and record the run."""
    if not isinstance(sample_id, str) or sample_id not in SEQUENCES:
        print('STOP: Use exactly "control", "A", "B", or "C" in quotation marks.')
        print("Fix only the sample_id line and run the same cell again.")
        return
    dna = SEQUENCES[sample_id]
    reference = SEQUENCES["control"]
    changes = [(i + 1, old, new) for i, (old, new)
               in enumerate(zip(reference, dna)) if old != new]
    peptide, stop_position, stop_codon = translate_coding_dna(dna)
    gc = 100 * (dna.count("G") + dna.count("C")) / len(dna)
    print(f"Sample: {sample_id}")
    print(f"DNA (coding, 5' to 3'): {triplets(dna)}")
    print(f"RNA (5' to 3'):        {triplets(dna.replace('T', 'U'))}")
    print(f"DNA length: {len(dna)} nt | GC: {gc:.1f}%")
    print(f"Base changes vs control: {len(changes)}")
    for position, old, new in changes:
        codon_index = (position - 1) // 3
        start = codon_index * 3
        old_codon = reference[start:start + 3].replace("T", "U")
        new_codon = dna[start:start + 3].replace("T", "U")
        print(f"Changed DNA base (counting from 1): {position}, {old} -> {new}")
        print(f"Changed RNA codon: {codon_index + 1}, {old_codon} -> {new_codon}")
    print("Predicted peptide: " + ("-".join(peptide) if peptide else "(none)"))
    print(f"Peptide length: {len(peptide)} amino acids (stop excluded)")
    print(f"First stop: codon {stop_position} ({stop_codon})"
          if stop_position is not None else "First stop: none in the supplied sequence")
    print("Scope: sequence prediction only; protein function was not measured.")
    RUNS[sample_id] = {"dna_length": len(dna), "gc": gc, "peptide": peptide,
                       "aa_length": len(peptide), "changes": changes,
                       "stop_position": stop_position, "stop_codon": stop_codon}


def check_work():
    missing = [label for label in ("control", "A", "B", "C") if label not in RUNS]
    if missing:
        print("Still needed: " + ", ".join(missing))
        print("Check sample_id in Cells 3-6, then run those cells and this check again.")
        print("Keep recording on your response sheet; a missing run is not lost written work.")
        return
    print("CHECK COMPLETE: control, A, B, and C have all been analyzed.")
    print(f"{'Sample':<9}{'DNA nt':<9}{'GC %':<9}{'Peptide':<14}{'AA count':<10}")
    for label in ("control", "A", "B", "C"):
        r = RUNS[label]
        print(f"{label:<9}{r['dna_length']:<9}{r['gc']:<9.1f}{r['peptide']:<14}{r['aa_length']:<10}")
    print("Compare with your response sheet. Classify the variants yourself.")


print("SETUP COMPLETE: 4 sequences loaded from lab02_sequences.fasta")
print("The > lines are sample labels; the following lines contain DNA.")
print(SEQUENCE_FILE.read_text(encoding="utf-8"))
print("Next: run Cell 3 (control). Do not edit this setup cell.")


SETUP COMPLETE: 4 sequences loaded from lab02_sequences.fasta
The > lines are sample labels; the following lines contain DNA.
>control
ATGGCTTTTCAAGAATGGCCTAAAGGCTAA
>A
ATGGCCTTTCAAGAATGGCCTAAAGGCTAA
>B
ATGGCTTTTCAAGTATGGCCTAAAGGCTAA
>C
ATGGCTTTTCAAGAATAGCCTAAAGGCTAA

Next: run Cell 3 (control). Do not edit this setup cell.


## Steps
### Cell 3 — Run the control (no edits)
Run the next cell. Copy its DNA length, GC percentage, peptide, and amino-acid count into the response-sheet table.

**Checkpoint:** `Sample: control`; 30 nucleotides; predicted peptide `M-A-F-Q-E-W-P-K-G`; 9 amino acids; first stop at codon 10. Stop codons are not counted in peptide length.


In [4]:
sample_id = "control"
analyze(sample_id)

Sample: control
DNA (coding, 5' to 3'): ATG GCT TTT CAA GAA TGG CCT AAA GGC TAA
RNA (5' to 3'):        AUG GCU UUU CAA GAA UGG CCU AAA GGC UAA
DNA length: 30 nt | GC: 40.0%
Base changes vs control: 0
Predicted peptide: M-A-F-Q-E-W-P-K-G
Peptide length: 9 amino acids (stop excluded)
First stop: codon 10 (UAA)
Scope: sequence prediction only; protein function was not measured.


### Cell 4 — Test variant A
**Before running:** predict whether the peptide will be unchanged, contain one different amino acid, or end earlier. Record your reason on the response sheet. This variant differs from the original control by **codon 2: GCT → GCC** (DNA base **6: T → C**).

Click inside the first line below. Replace only the word `control` inside the quotation marks with **`A`**. Leave the second line unchanged. Your first line must read:
```python
sample_id = "A"
```
Now run this cell. **Check the output says `Sample: A` and `Base changes vs control: 1`.** If it still says control, correct the first line and run this cell again.

Copy the result to row A on your response sheet. Classify the variant by comparing with **control**, not with the previous variant. Each variant is a separate one-base change from control; the changes are not cumulative.


In [6]:
sample_id = "A"
analyze(sample_id)

Sample: A
DNA (coding, 5' to 3'): ATG GCC TTT CAA GAA TGG CCT AAA GGC TAA
RNA (5' to 3'):        AUG GCC UUU CAA GAA UGG CCU AAA GGC UAA
DNA length: 30 nt | GC: 43.3%
Base changes vs control: 1
Changed DNA base (counting from 1): 6, T -> C
Changed RNA codon: 2, GCU -> GCC
Predicted peptide: M-A-F-Q-E-W-P-K-G
Peptide length: 9 amino acids (stop excluded)
First stop: codon 10 (UAA)
Scope: sequence prediction only; protein function was not measured.


### Cell 5 — Test variant B
**Before running:** predict whether the peptide will be unchanged, contain one different amino acid, or end earlier. Record your reason on the response sheet. This variant differs from the original control by **codon 5: GAA → GTA** (DNA base **14: A → T**).

Click inside the first line below. Replace only the word `control` inside the quotation marks with **`B`**. Leave the second line unchanged. Your first line must read:
```python
sample_id = "B"
```
Now run this cell. **Check the output says `Sample: B` and `Base changes vs control: 1`.** If it still says control, correct the first line and run this cell again.

Copy the result to row B on your response sheet. Classify the variant by comparing with **control**, not with the previous variant. Each variant is a separate one-base change from control; the changes are not cumulative.


In [7]:
sample_id = "B"
analyze(sample_id)

Sample: B
DNA (coding, 5' to 3'): ATG GCT TTT CAA GTA TGG CCT AAA GGC TAA
RNA (5' to 3'):        AUG GCU UUU CAA GUA UGG CCU AAA GGC UAA
DNA length: 30 nt | GC: 40.0%
Base changes vs control: 1
Changed DNA base (counting from 1): 14, A -> T
Changed RNA codon: 5, GAA -> GUA
Predicted peptide: M-A-F-Q-V-W-P-K-G
Peptide length: 9 amino acids (stop excluded)
First stop: codon 10 (UAA)
Scope: sequence prediction only; protein function was not measured.


### Cell 6 — Test variant C
**Before running:** predict whether the peptide will be unchanged, contain one different amino acid, or end earlier. Record your reason on the response sheet. This variant differs from the original control by **codon 6: TGG → TAG** (DNA base **17: G → A**).

Click inside the first line below. Replace only the word `control` inside the quotation marks with **`C`**. Leave the second line unchanged. Your first line must read:
```python
sample_id = "C"
```
Now run this cell. **Check the output says `Sample: C` and `Base changes vs control: 1`.** If it still says control, correct the first line and run this cell again.

Copy the result to row C on your response sheet. Classify the variant by comparing with **control**, not with the previous variant. Each variant is a separate one-base change from control; the changes are not cumulative.


In [8]:
sample_id = "C"
analyze(sample_id)

Sample: C
DNA (coding, 5' to 3'): ATG GCT TTT CAA GAA TAG CCT AAA GGC TAA
RNA (5' to 3'):        AUG GCU UUU CAA GAA UAG CCU AAA GGC UAA
DNA length: 30 nt | GC: 36.7%
Base changes vs control: 1
Changed DNA base (counting from 1): 17, G -> A
Changed RNA codon: 6, UGG -> UAG
Predicted peptide: M-A-F-Q-E
Peptide length: 5 amino acids (stop excluded)
First stop: codon 6 (UAG)
Scope: sequence prediction only; protein function was not measured.


## Checks
### Cell 7 — Check the four completed analyses
Run after testing A, B, and C. **Expected:** `CHECK COMPLETE` and four rows. If a label is missing, fix and rerun the corresponding cell; do not delete your written results.


In [9]:
check_work()

CHECK COMPLETE: control, A, B, and C have all been analyzed.
Sample   DNA nt   GC %     Peptide       AA count  
control  30       40.0     MAFQEWPKG     9         
A        30       43.3     MAFQEWPKG     9         
B        30       40.0     MAFQVWPKG     9         
C        30       36.7     MAFQE         5         
Compare with your response sheet. Classify the variants yourself.


## Interpret and finish
Complete Q1-Q4 on page 6 of the handout. Use exact codons or output values as evidence; do not equate a sequence prediction with a measurement of biological function.

**Required submission:** each student submits their own completed response sheets (handout pages 5-6), as the edited Word document or clear page images. A plain document with the same headings is also acceptable. Mark whether your evidence came from your own run, a partner's run, an instructor demonstration, or the supplied backup. If Canvas fails, keep a local copy and show/hand the work to the instructor before leaving.

**Keep your notebook when possible:** use File → Download → Download .ipynb (or the equivalent notebook download entry under File). Keep `.ipynb`, not `.py`. It is a reusable record, not a second required submission today. No GitHub upload is required during this session.

### If something goes wrong
| What you see | Do this | If it persists |
|---|---|---|
| No Run button | Click in the code cell, not an explanation or the AI panel; press Shift+Enter. | Ask a partner to point to the cell. |
| `NameError` / `analyze` is not defined | Run Cell 2, then rerun the current analysis cell. | Reopen the starter; after two minutes switch to partner/backup. |
| `SyntaxError` or quote error | Replace only that short cell with its two supplied lines. Retype plain quotation marks in Colab. | Restore the cell from the starter. |
| `STOP: Use exactly...` | Use `"control"`, `"A"`, `"B"`, or `"C"`; case matters. | Ask a partner to compare the first line. |
| Output still says control | Change the first line in the correct cell and rerun it. | Use the expected sample label as your checkpoint. |
| Runtime disconnected | Reconnect, rerun Cell 2, then Cells 3-6 with their correct labels. | Do not spend more than two minutes; use partner/backup. |
| Cannot edit or save | Try File → Save a copy in Drive, then the upload route. | Keep answers outside Colab and switch route. |

## Next steps
Keep the notebook. A later lab can read real sequence files and compare more variants. Today's translation tool is a deliberately limited, deterministic model—not an AI prediction of protein function.

### Sources and provenance
All four DNA sequences, exercises, and expected numerical outputs were constructed for this lab. Code uses the **NCBI standard genetic code (table 1)**. No external dataset or patient sequence is used.
- [NCBI: The Genetic Codes](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/data-processing/taxonomy-processing/genetic-codes/)
- [NCBI: FASTA Format](https://www.ncbi.nlm.nih.gov/genbank/fastaformat/)
- [MedlinePlus Genetics: Kinds of gene variants](https://medlineplus.gov/genetics/understanding/mutationsanddisorders/possiblemutations/)
- [Google: Colab FAQ](https://research.google.com/colaboratory/faq.html)

Prepared September 28, 2026. Student outputs are intentionally blank before use so predictions can precede execution.
